# 10 — TF-IDF from Formula to scikit-learn

**Learning objective.** Derive TF-IDF intuition, inspect sparse weights, and connect weighting to information value.

This notebook is intentionally **offline-reproducible**: the examples use local data or deterministic toy corpora so the rendered GitHub output can be trusted without hidden API calls. The focus is always **concept → inspectable representation → library implementation → result → failure modes → production implication**.

In [1]:
from pathlib import Path
import re, math, json, random, statistics
import numpy as np
import pandas as pd
np.random.seed(42)
random.seed(42)
pd.set_option('display.max_colwidth', 120)
DATA = Path('data')
print('Reproducibility seed: 42')

Reproducibility seed: 42


For term $t$ in document $d$:

$$\mathrm{tfidf}(t,d)=\mathrm{tf}(t,d)\times\log\frac{N}{\mathrm{df}(t)}$$

Libraries often use smoothing and vector normalization. The key intuition is stable: **a term gets more weight when it is frequent in this document but uncommon across the corpus**.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
docs=['refund delayed refund','refund completed','login delayed']
vec=TfidfVectorizer(norm=None,smooth_idf=False)
X=vec.fit_transform(docs)
pd.DataFrame(np.round(X.toarray(),3),columns=vec.get_feature_names_out())

   completed  delayed  login  refund
0      0.000    1.405  0.000   2.811
1      2.099    0.000  0.000   1.405
2      0.000    1.405  2.099   0.000

In [3]:
terms=vec.get_feature_names_out()
idf=dict(zip(terms,np.round(vec.idf_,3)))
print('IDF:',idf)
print('Most discriminative in doc1:', terms[X.toarray()[0].argmax()])

IDF: {'completed': np.float64(2.099), 'delayed': np.float64(1.405), 'login': np.float64(2.099), 'refund': np.float64(1.405)}
Most discriminative in doc1: refund


In [4]:

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7,3))
im=ax.imshow(X.toarray(),aspect='auto')
ax.set_xticks(range(len(terms))); ax.set_xticklabels(terms,rotation=45,ha='right')
ax.set_yticks(range(len(docs))); ax.set_yticklabels([f'doc{i+1}' for i in range(len(docs))])
ax.set_title('TF-IDF document–term matrix')
fig.colorbar(im,ax=ax,label='weight')
plt.tight_layout(); plt.show()


[static visualization generated successfully during execution; rerun in Jupyter/VS Code to display]


---
    ## Production takeaways
    - Preserve preprocessing as part of the model contract; training/inference skew is an NLP failure mode, not an implementation detail.
    - Inspect intermediate representations rather than treating tokenizers/vectorizers/models as black boxes.
    - Prefer the simplest representation/model that meets quality, latency, governance, and maintenance requirements.

### What you should now be able to explain
- Explain term frequency vs inverse document frequency
- Inspect a sparse representation instead of treating it as opaque